# Lezione 4: Proporzioni e relazioni — Pie, Box, Scatter e Bubble plot

## Perché questo ti riguarda?

Non tutte le domande sui dati sono "quanto?" o "come cambia nel tempo?". A volte la domanda vera è:

- Che **proporzione** del totale spetta a ogni gruppo? → **pie chart**
- Come si **distribuiscono** i miei dati e c'è qualche valore strano tra loro? → **box plot**
- Esiste una **relazione** tra due variabili diverse? → **scatter plot**
- Quella relazione cambia se consideriamo anche una **terza variabile**? → **bubble plot**

Questi quattro strumenti chiudono il cerchio dell'analisi esplorativa: sappiamo già raccontare le tendenze (linee), confrontare categorie (barre) e vedere le distribuzioni (istogrammi). Ora impariamo a vedere **ripartizione, dispersione e relazione**.

Continuiamo a lavorare con `df_can` (immigrazione verso il Canada, 1980-2013), la lista `years` e l'indice per paese.

## Preparazione dell'ambiente e dei dati

In [ ]:
# Import primary modules
import numpy as np   # utile per il calcolo scientifico (lo useremo per np.polyfit)
import pandas as pd  # libreria principale di strutture dati

# Importing Matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('ggplot')  # opzionale: stile visivo tipo ggplot

print('Matplotlib version: ', mpl.__version__)

Scarichiamo il dataset già pulito dell'immigrazione verso il Canada (lo stesso file usato nelle lezioni precedenti).

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Dati caricati in un DataFrame di pandas!')
df_can.head()

In [ ]:
# Fissiamo 'Country' come indice per poter cercare i paesi con .loc
df_can.set_index('Country', inplace=True)

# Lista degli anni 1980-2013 come stringhe (nomi delle colonne)
years = list(map(str, range(1980, 2014)))

print('Dimensioni:', df_can.shape)
df_can.head()

## Pie chart: dividere la pizza

Un pie chart è letteralmente una **pizza** che dividi tra vari commensali. Ogni fetta è proporzionale all'appetito (o al peso) di ogni commensale rispetto al totale. È lo strumento giusto quando vuoi mostrare **di cosa è fatto il 100%**, non quando vuoi confrontare grandi magnitudini assolute (per quello, usa le barre).

Vediamo che proporzione dell'immigrazione totale verso il Canada (1980-2013) è venuta da ogni continente. Per prima cosa raggruppiamo con `groupby` (dividi → applica → combina: divide il dataframe in gruppi in base a `Continent`, applica `.sum()` a ogni gruppo, e combina i risultati).

In [ ]:
# raggruppare i paesi per continente e sommare i loro valori
df_continents = df_can.groupby('Continent').sum(numeric_only=True)
df_continents.head()

In [ ]:
df_continents['Total'].plot(kind='pie',
                             figsize=(10, 6),
                             autopct='%1.1f%%',   # mostra la percentuale dentro ogni fetta
                             startangle=90,        # inizia a disegnare da 90°
                             shadow=True)           # ombra per dare sensazione di 3D

plt.title('Immigrazione in Canada per continente (1980-2013)')
plt.axis('equal')  # forza il cerchio ad apparire come cerchio, non come ovale
plt.legend(labels=df_continents.index, loc='upper left')

plt.show()

Se le percentuali si accumulano o si sovrappongono alle etichette, le separiamo con `pctdistance` e usiamo `explode` per "estrarre" una fetta verso l'esterno e attirare l'attenzione su di essa (come tirare fuori una fetta di pizza dal centro del piatto).

In [ ]:
explode_list = [0.1, 0, 0, 0, 0.1, 0.1]  # quanto si separa ogni fetta dal centro (0 = non si muove)

df_continents['Total'].plot(kind='pie',
                             figsize=(10, 6),
                             autopct='%1.1f%%',
                             startangle=90,
                             shadow=True,
                             labels=None,          # togliamo le etichette sopra il pie
                             pctdistance=1.12,     # spinge la percentuale fuori dalla fetta
                             colors=['gold', 'yellowgreen', 'lightcoral', 'lightskyblue', 'lightgreen', 'pink'],
                             explode=explode_list)

plt.title('Immigrazione in Canada per continente (1980-2013)', y=1.12, fontsize=15)
plt.axis('equal')
plt.legend(labels=df_continents.index, loc='upper left', fontsize=7)

plt.show()

## Box plot: la mediana e i suoi quarti

Immagina i voti di un esame di 30 studenti ordinati dal più basso al più alto. Il voto che rimane esattamente nel mezzo è la **mediana**. Se dividi quella lista ordinata in quattro parti uguali, i punti di taglio sono i **quartili**: il primo quartile (Q1) lascia il 25% dei voti sotto, il terzo quartile (Q3) lascia il 75% sotto. Un box plot disegna esattamente questo: una scatola che va da Q1 a Q3 (con la mediana segnata all'interno), e dei "baffi" che si estendono fino al minimo e al massimo ragionevoli.

Vediamo il numero di immigrati arrivati dal Giappone, 1980-2013.

In [ ]:
# estrarlo come dataframe (con doppia parentesi quadra) per poter usare describe() dopo
df_japan = df_can.loc[['Japan'], years].transpose()

df_japan.plot(kind='box', figsize=(8, 6))

plt.title('Box plot degli immigrati giapponesi (1980-2013)')
plt.ylabel('Numero di immigrati')

plt.show()

# vedere i numeri esatti dietro alla scatola
df_japan.describe()

A colpo d'occhio si legge: il minimo si aggira sui 200, il massimo sui 1300, la mediana sui 900; il 25% degli anni ha avuto 500 immigrati o meno, il 75% ne ha avuti 1100 o meno.

### Outlier: i valori che escono dalla norma

Un box plot non riassume soltanto: **rileva anche i valori anomali** automaticamente. La regola è matematica, non arbitraria. Per prima cosa si calcola il range interquartile:

```
IQR = Q3 - Q1
```

E un valore viene segnato come **outlier** se:

- è maggiore di `Q3 + 1.5 × IQR`, oppure
- è minore di `Q1 - 1.5 × IQR`

Ad esempio, se confrontiamo le 15 principali fonti di immigrazione raggruppate per decenni e nel decennio 2000 abbiamo Q1 = 36.101,5 e Q3 = 105.505,5, l'IQR è 69.404. Qualsiasi paese con più di `105.505,5 + 1,5 × 69.404 = 209.611,5` immigrati in quel decennio appare nel grafico come un punto isolato sopra il baffo superiore: un outlier reale, non un errore di calcolo. In questo dataset, **Cina e India** superano quella soglia.

Confrontiamo entrambi i paesi nello stesso box plot, disegnato orizzontale con `vert=False`.

In [ ]:
df_CI = df_can.loc[['China', 'India'], years].transpose()

df_CI.plot(kind='box', figsize=(10, 7), color='blue', vert=False)

plt.title('Box plot degli immigrati da Cina e India (1980-2013)')
plt.xlabel('Numero di immigrati')

plt.show()

df_CI.describe()

### Subplot: più grafici, un'unica figura

Quando vogliamo confrontare due viste dello stesso dato affiancate (il box plot e la linea temporale di Cina e India), usiamo i **subplot**. Lavoriamo con l'artist layer (`fig`, `ax`) invece che con `plt` diretto: il trucco è passare `ax=ax0` o `ax=ax1` al metodo `.plot()` di ogni dataframe, per dire a *pandas* su quale sottografico disegnare.

In [ ]:
fig = plt.figure(figsize=(20, 6))

ax0 = fig.add_subplot(1, 2, 1)  # 1 riga, 2 colonne, primo grafico
ax1 = fig.add_subplot(1, 2, 2)  # 1 riga, 2 colonne, secondo grafico

df_CI.plot(kind='box', color='blue', vert=False, ax=ax0)
ax0.set_title('Box plot di Cina e India (1980-2013)')
ax0.set_xlabel('Numero di immigrati')

df_CI.plot(kind='line', ax=ax1)
ax1.set_title('Evoluzione annuale di Cina e India (1980-2013)')
ax1.set_ylabel('Numero di immigrati')
ax1.set_xlabel('Anni')

plt.show()

## Scatter plot: la relazione tra due variabili

Uno scatter plot risponde a una domanda diversa: esiste una relazione tra due variabili numeriche? Pensa all'esempio classico di altezza e peso di un gruppo di persone: se disegni un punto per persona (altezza in x, peso in y), vedrai se tendono a salire insieme. A differenza di un grafico a linee, i punti di uno scatter **non si collegano tra loro**: ogni punto è un'osservazione indipendente, e il pattern (se esiste) emerge dalla nuvola di punti, non da una linea che li unisce.

Vediamo la relazione tra l'anno e il totale di immigrati in Canada.

In [ ]:
# sommare tutti i paesi per ogni anno
df_tot = pd.DataFrame(df_can[years].sum(axis=0))
df_tot.index = map(int, df_tot.index)   # convertire l'anno in intero, ci servirà per la regressione
df_tot.reset_index(inplace=True)
df_tot.columns = ['year', 'total']

df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')

plt.title('Immigrazione totale in Canada (1980-2013)')
plt.xlabel('Anno')
plt.ylabel('Numero di immigrati')

plt.show()

Si vede una tendenza ascendente chiara. Uno scatter plot ci permette di **vedere** la relazione, ma se vogliamo quantificarla e usarla per prevedere, ci serve una regressione.

### Regressione lineare manuale con `np.polyfit`

`np.polyfit(x, y, deg=1)` calcola, tramite minimi quadrati, i coefficienti della retta che meglio si adatta ai punti (`deg=1` significa grado 1: una linea retta; `deg=2` sarebbe una parabola, e così via).

In [ ]:
x = df_tot['year']
y = df_tot['total']

fit = np.polyfit(x, y, deg=1)
fit  # array([pendenza, intercetta])

Il risultato sono i coefficienti di `y = pendenza * x + intercetta`. Disegniamo la retta sopra lo scatter con `plt.plot`.

In [ ]:
df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')

plt.title('Immigrazione totale in Canada (1980-2013)')
plt.xlabel('Anno')
plt.ylabel('Numero di immigrati')

# la retta di regressione
plt.plot(x, fit[0] * x + fit[1], color='red')
plt.annotate('y={0:.0f} x + {1:.0f}'.format(fit[0], fit[1]), xy=(2000, 150000))

plt.show()

# previsione semplice per l'anno 2015
prediccion_2015 = fit[0] * 2015 + fit[1]
print(f'Previsione di immigrati totali per il 2015: {prediccion_2015:,.0f}')

Con questa equazione possiamo fare una previsione semplice: sostituendo `year=2015` otteniamo una stima di immigrati per quell'anno, che potremmo confrontare con il dato reale e vedere quanto bene (o male) una linea retta si adatta alla realtà.

Fare questo a mano con `np.polyfit` è importante per una ragione precisa: ci obbliga a capire cos'è una regressione — una retta che minimizza l'errore totale — prima di usare, più avanti, librerie come Seaborn che la calcolano e la disegnano con una sola chiamata (`sns.regplot`, ad esempio).

## Bubble plot: scatter plot con una terza storia

Un bubble plot è uno scatter plot a cui aggiungiamo una dimensione in più: anche la **dimensione** di ogni punto significa qualcosa. Se lo scatter normale dice "dove si trova ogni dato in x e y", il bubble plot aggiunge "e inoltre, quanto è grande quel dato in una terza variabile z". È come uno scatter di altezza contro peso dove, inoltre, la dimensione di ogni punto rappresenta l'età della persona: a colpo d'occhio vediamo tre variabili contemporaneamente.

Confrontiamo l'immigrazione da **Brasile e Argentina**, usando la dimensione della bolla per rappresentare la grandezza di ogni anno.

Per prima cosa, prepariamo i dati trasposti con l'anno come colonna.

In [ ]:
df_can_t = df_can[years].transpose()
df_can_t.index = map(int, df_can_t.index)
df_can_t.index.name = 'Year'
df_can_t.reset_index(inplace=True)

df_can_t.head()

### Normalizzare la dimensione

I valori grezzi di immigrati non servono direttamente come dimensione del punto: alcuni sono troppo piccoli per essere visti e altri troppo grandi per stare sullo schermo. Per questo normalizziamo ogni serie nell'intervallo [0, 1] con il *feature scaling*: il valore minimo diventa 0, il massimo diventa 1, e tutto il resto si distribuisce proporzionalmente in mezzo.

In [ ]:
# normalizzare Brasile
norm_brazil = (df_can_t['Brazil'] - df_can_t['Brazil'].min()) / (df_can_t['Brazil'].max() - df_can_t['Brazil'].min())

# normalizzare Argentina
norm_argentina = (df_can_t['Argentina'] - df_can_t['Argentina'].min()) / (df_can_t['Argentina'].max() - df_can_t['Argentina'].min())

norm_brazil.head()

Poiché il risultato normalizzato vive tra 0 e 1, se lo usiamo così com'è come dimensione del punto (`s`) quasi non si vedrebbe nel grafico. Per questo lo scaliamo: lo moltiplichiamo per un fattore grande e ci sommiamo un minimo affinché anche il valore 0 appaia come un punto piccolo, non invisibile.

In [ ]:
# Brasile
ax0 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='Brazil',
                     figsize=(14, 8),
                     alpha=0.5,
                     color='green',
                     s=norm_brazil * 2000 + 10,   # dimensione = normalizzato * 2000 + 10 (perché non scompaia)
                     xlim=(1975, 2015))

# Argentina, sugli stessi assi di Brasile (ax=ax0)
ax1 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='Argentina',
                     alpha=0.5,
                     color='blue',
                     s=norm_argentina * 2000 + 10,
                     ax=ax0)

ax0.set_ylabel('Numero di immigrati')
ax0.set_title('Immigrazione da Brasile e Argentina (1980-2013)')
ax0.legend(['Brasile', 'Argentina'], loc='upper left', fontsize='x-large')

plt.show()

Il risultato mostra bolle più grandi durante la crisi argentina del 1998-2002 e durante la crisi del real brasiliano nel 1999: la **dimensione della bolla racconta, letteralmente, la grandezza di ogni crisi migratoria** anno per anno, qualcosa che uno scatter piatto non potrebbe trasmettere.

## Punti chiave

- Il **pie chart** mostra **proporzioni di un totale**; usa `explode` per evidenziare una fetta e `autopct` per mostrare la percentuale direttamente sul grafico.
- Il **box plot** riassume una distribuzione in cinque numeri (minimo, Q1, mediana, Q3, massimo) e **rileva outlier automaticamente** con la regola dell'1,5 × IQR.
- Lo **scatter plot** rivela **relazioni tra due variabili numeriche**; i punti non si collegano tra loro.
- `np.polyfit(x, y, deg=1)` calcola a mano la retta di regressione; capirlo ora prepara a usare strumenti di regressione automatici più avanti.
- Il **bubble plot** è uno scatter plot con una terza variabile codificata nella **dimensione** del punto; normalizza e riscala sempre quella variabile prima di passarla al parametro `s`.
- I **subplot** (`fig.add_subplot()`) permettono di confrontare più viste dello stesso dato in un'unica figura.

## Prossimo passo

Nella pratica applicherai questi quattro strumenti sullo stesso dataset `df_can`, con nuovi ritagli di dati (anno 2013, Cina vs. India).